#Initialization

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab

Mounted at /content/drive
/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab


In [2]:
import os
SEED = 123
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # must be before torch import

In [3]:
import math, random, hashlib, copy
import pandas as pd
import numpy as np

from pathlib import Path
from typing  import Tuple, List
from PIL     import Image, ImageEnhance

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from sklearn.metrics import fbeta_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

#Data Loading

In [4]:
train_sc4a_df = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc4a/train_sc4a.xlsx")
val_sc4a_df   = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc4a/val_sc4a.xlsx")
test_sc4a_df  = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc4a/test_sc4a.xlsx")

train_sc4b_df = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc4b/train_sc4b.xlsx")
val_sc4b_df   = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc4b/val_sc4b.xlsx")
test_sc4b_df  = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc4b/test_sc4b.xlsx")

In [5]:
# Split Summarize Data

def split_hash(ids):
    txt = "\n".join(sorted(map(str, ids)))
    return hashlib.sha256(txt.encode("utf-8")).hexdigest()

def summarize_split(df, name):
    child_labels = (
        df.groupby("child_id")["label"]
          .agg(lambda s: int(pd.Series.mode(s).iloc[0]))
          .value_counts()
          .sort_index()
          .to_dict()
    )
    label_counts = df["label"].value_counts().sort_index().to_dict()
    return {
        "split": name,
        "n_images": int(len(df)),
        "n_children": int(df["child_id"].nunique()),
        "child_label_0": int(child_labels.get(0, 0)),
        "child_label_1": int(child_labels.get(1, 0)),
        "image_label_0": int(label_counts.get(0, 0)),
        "image_label_1": int(label_counts.get(1, 0)),
        "child_id_hash": split_hash(df["child_id"].unique().tolist()),
    }

##Summary Sc.4.a. Dataset

In [6]:
split_summary = pd.DataFrame([
    summarize_split(train_sc4a_df, "train"),
    summarize_split(val_sc4a_df, "val"),
    summarize_split(test_sc4a_df, "test"),
])

print("=== SUBJECT-DISJOINT SPLIT SUMMARY ===")
display(split_summary.iloc[:,:-1])

=== SUBJECT-DISJOINT SPLIT SUMMARY ===


,split,n_images,n_children,child_label_0,child_label_1,image_label_0,image_label_1
0,train,608,152,87,65,348,260
1,val,19,19,11,8,11,8
2,test,19,19,11,8,11,8


##Summary Sc.4.b. Dataset

In [7]:
split_summary = pd.DataFrame([
    summarize_split(train_sc4b_df, "train"),
    summarize_split(val_sc4b_df, "val"),
    summarize_split(test_sc4b_df, "test"),
])

print("=== SUBJECT-DISJOINT SPLIT SUMMARY ===")
display(split_summary.iloc[:,:-1])

=== SUBJECT-DISJOINT SPLIT SUMMARY ===


,split,n_images,n_children,child_label_0,child_label_1,image_label_0,image_label_1
0,train,152,152,87,65,87,65
1,val,19,19,11,8,11,8
2,test,19,19,11,8,11,8


#Pre-processing Set

In [8]:
EPOCHS = 20

In [9]:
# Transformation

IMAGE_SIZE = 224

imagenet_norm = transforms.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225])

train_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    imagenet_norm,
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    imagenet_norm,
])

In [10]:
# Dataset Loader

BATCH_SIZE = 32
NUM_WORKERS = 2

class FaceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        sample = self.dataframe.iloc[idx]
        image  = Image.open(sample["path"]).convert("RGB")
        image  = self.transform(image) if self.transform else transforms.ToTensor()(image)
        label  = int(sample["label"])
        return {
            "image": image,
            "label": torch.tensor(label, dtype=torch.long),
            "child_id": str(sample["child_id"]),
            "path": str(sample["path"]),
        }

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loader(dataframe, transform, shuffle):
    ds = FaceDataset(dataframe, transform=transform)
    g  = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        generator=g,
        worker_init_fn=seed_worker,  # ← ADD THIS
    )

#EfficientNet-B0

In [11]:
sc4_train_result = []

In [12]:
# Model - pre-trained EfficientNet B0

def create_model(num_classes=2):
    weights = EfficientNet_B0_Weights.DEFAULT
    model   = efficientnet_b0(weights=weights)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model

In [13]:
# Modeling: Train per-epoch Function

def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train(mode=train_mode)

    total_loss, total_correct, total_count = 0.0, 0, 0
    all_preds, all_labels = [], []
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train_mode):
            logits = model(images)
            loss = criterion(logits, labels)

        if train_mode:
            loss.backward()
            optimizer.step()

        preds = logits.argmax(1)
        total_loss += loss.item() * images.size(0)
        total_correct += (preds == labels).sum().item()
        total_count += images.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    avg_loss = total_loss / max(1, total_count)
    avg_acc  = total_correct / max(1, total_count)
    f2       = fbeta_score(all_labels, all_preds, beta=2, pos_label=1, zero_division=0)

    tn = ((all_preds == 0) & (all_labels == 0)).sum()
    fp = ((all_preds == 1) & (all_labels == 0)).sum()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return avg_loss, avg_acc, f2, specificity

#Sc.4.a. Model Development

##Sc.4.a. EfficientNet-B0 with SGD

###Sc.4.a. EfficientNet-B0 with SGD Class Weight CE Loss Function

In [14]:
criterion = nn.CrossEntropyLoss(weight=None)

In [15]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
#torch.use_deterministic_algorithms(True)

model = create_model(num_classes=2).to(device)


Device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 151MB/s]


In [16]:
# Class Weight
train_counts = train_sc4a_df["label"].value_counts().sort_index().to_dict()
n0 = train_counts.get(0, 1)
n1 = train_counts.get(1, 1)

# Rare class (1) gets HIGH weight, frequent class (0) gets LOW weight
base_class_weights = torch.tensor([1.0 / n0, 1.0 / n1], dtype=torch.float32, device=device)

In [17]:
criterion = nn.CrossEntropyLoss(weight=base_class_weights)

###Sc.4.a. SGD Optimizer

In [18]:
# SGD
LR          = 1e-3
MOMENTUM    = 0.9
WEIGHT_DECAY= 1e-4
NESTEROV    = True

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    nesterov=NESTEROV
    )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

###Sc.4.a. EfficientNet-B0 with SGD Training

In [19]:
# Load Dataset

dl_train_sc4a = make_loader(train_sc4a_df, train_tfms, shuffle=True)
dl_val_sc4a   = make_loader(val_sc4a_df, eval_tfms, shuffle=False)
dl_test_sc4a  = make_loader(test_sc4a_df, eval_tfms, shuffle=False)

print(f"\ntrain={len(train_sc4a_df)} | val={len(val_sc4a_df)} | test={len(test_sc4a_df)}")


train=608 | val=19 | test=19


In [20]:
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc4a_sgd.pth"

In [21]:
# Training Loop

print("Optimizer     :", optimizer)
print("Loss Function :", criterion)
print("Loss Weight   :", criterion.weight)

# ---------------------------------------------------------
# Best-model tracking
# ---------------------------------------------------------

best_val_f2   = -1.0
best_val_loss = float("inf")
best_epoch    = None
best_state    = None

best_val_f2_constrained   = -1.0
best_val_f2_unconstrained = -1.0
best_val_loss_constrained   = float("inf")
best_val_loss_unconstrained = float("inf")
best_state_constrained    = None
best_state_unconstrained  = None
best_epoch_constrained    = None
best_epoch_unconstrained  = None

MIN_SPECIFICITY = 0.30
MIN_DELTA = 1e-4

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_acc, tr_f2, tr_spec = run_epoch(model, dl_train_sc4a, criterion, optimizer=optimizer)
    val_loss, val_acc, val_f2, val_spec = run_epoch(model, dl_val_sc4a, criterion, optimizer=None)
    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {tr_loss:.4f} acc {tr_acc:.4f} "
        f"f2 {tr_f2:.4f} spec {tr_spec:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} "
        f"f2 {val_f2:.4f} spec {val_spec:.4f}"
    )

    meets_specificity = val_spec >= MIN_SPECIFICITY

    is_better_unconstrained = (
        val_f2 > best_val_f2_unconstrained + MIN_DELTA
    ) or (
        np.isclose(val_f2, best_val_f2_unconstrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_unconstrained
    )

    if is_better_unconstrained:
        best_val_f2_unconstrained = val_f2
        best_val_loss_unconstrained = val_loss
        best_epoch_unconstrained = epoch
        best_state_unconstrained = copy.deepcopy(model.state_dict())

    is_better_constrained = meets_specificity and (
        val_f2 > best_val_f2_constrained + MIN_DELTA
    ) or (
        meets_specificity
        and np.isclose(val_f2, best_val_f2_constrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_constrained
    )

    if is_better_constrained:
        best_val_f2_constrained = val_f2
        best_val_loss_constrained = val_loss
        best_epoch_constrained = epoch
        best_state_constrained = copy.deepcopy(model.state_dict())

    if best_state_constrained is not None:
        best_state = best_state_constrained
        best_epoch = best_epoch_constrained
        best_val_f2 = best_val_f2_constrained
        best_val_loss = best_val_loss_constrained
        selection_mode = "constrained (specificity met)"
    else:
        best_state = best_state_unconstrained
        best_epoch = best_epoch_unconstrained
        best_val_f2 = best_val_f2_unconstrained
        best_val_loss = best_val_loss_unconstrained

        selection_mode = ("unconstrained (fallback — no epoch met specificity)")


if best_state is not None:
    #model.load_state_dict(best_state)
    torch.save(best_state,BEST_PATH)

    print("\nModel selected via:", selection_mode)
    print(f"Best epoch: {best_epoch} | val_f2: {best_val_f2:.4f} | val_loss: {best_val_loss:.4f}")
else:
    raise RuntimeError("No valid model state was recorded.")

sc4_train_result.append([
    "sc4a-efficientnet-b0-sgd",
    "sc4a", "efficientnet-b0", "sgd",
    selection_mode,
    best_epoch,
    best_val_f2,
    ])

Optimizer     : SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.001
    maximize: False
    momentum: 0.9
    nesterov: True
    weight_decay: 0.0001
)
Loss Function : CrossEntropyLoss()
Loss Weight   : tensor([0.0029, 0.0038], device='cuda:0')
Epoch 01/20 | train loss 0.6817 acc 0.5493 f2 0.6016 spec 0.4799 | val loss 0.7668 acc 0.3158 f2 0.1282 spec 0.4545
Epoch 02/20 | train loss 0.6535 acc 0.6530 f2 0.5660 spec 0.7241 | val loss 0.7753 acc 0.3158 f2 0.1282 spec 0.4545
Epoch 03/20 | train loss 0.6208 acc 0.7089 f2 0.6352 spec 0.7701 | val loss 0.7837 acc 0.3684 f2 0.1316 spec 0.5455
Epoch 04/20 | train loss 0.5690 acc 0.7747 f2 0.8156 spec 0.7155 | val loss 0.7990 acc 0.3684 f2 0.1316 spec 0.5455
Epoch 05/20 | train loss 0.5431 acc 0.8076 f2 0.7869 spec 0.8190 | val loss 0.8247 acc 0.3684 f2 0.1316 spec 0.5455
Epoch 06/20 | train loss 0.5034 acc 0.8536 f2 0.8429 spec 0.8563 | val loss 0.8456 acc 0.4

##Sc.4.a. EfficientNet-B0 with Adam

###Sc.4.a. EfficientNet-B0 with Adam Class Weight CE Loss Function

In [22]:
criterion = nn.CrossEntropyLoss(weight=None)

In [23]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
#torch.use_deterministic_algorithms(True)

model = create_model(num_classes=2).to(device)


Device: cuda


In [24]:
# Class Weight
train_counts = train_sc4a_df["label"].value_counts().sort_index().to_dict()
n0 = train_counts.get(0, 1)
n1 = train_counts.get(1, 1)

# Rare class (1) gets HIGH weight, frequent class (0) gets LOW weight
base_class_weights = torch.tensor([1.0 / n0, 1.0 / n1], dtype=torch.float32, device=device)

In [25]:
criterion = nn.CrossEntropyLoss(weight=base_class_weights)

###Sc.4.a. Adam Optimizer

In [26]:
# Adam
LR          = 1e-4
BETAS       = (0.9, 0.999)
EPS         = 1e-8
WEIGHT_DECAY= 0.0

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    betas=BETAS,
    eps=EPS,
    weight_decay=WEIGHT_DECAY
    )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

###Sc.4.a. EfficientNet-B0 with Adam Training

In [27]:
# Load Dataset

dl_train_sc4a = make_loader(train_sc4a_df, train_tfms, shuffle=True)
dl_val_sc4a   = make_loader(val_sc4a_df, eval_tfms, shuffle=False)
dl_test_sc4a  = make_loader(test_sc4a_df, eval_tfms, shuffle=False)

print(f"\ntrain={len(train_sc4a_df)} | val={len(val_sc4a_df)} | test={len(test_sc4a_df)}")


train=608 | val=19 | test=19


In [28]:
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc4a_adam.pth"

In [29]:
# Training Loop

print("Optimizer     :", optimizer)
print("Loss Function :", criterion)
print("Loss Weight   :", criterion.weight)

# ---------------------------------------------------------
# Best-model tracking
# ---------------------------------------------------------

best_val_f2   = -1.0
best_val_loss = float("inf")
best_epoch    = None
best_state    = None

best_val_f2_constrained   = -1.0
best_val_f2_unconstrained = -1.0
best_val_loss_constrained   = float("inf")
best_val_loss_unconstrained = float("inf")
best_state_constrained    = None
best_state_unconstrained  = None
best_epoch_constrained    = None
best_epoch_unconstrained  = None

MIN_SPECIFICITY = 0.30
MIN_DELTA = 1e-4

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_acc, tr_f2, tr_spec = run_epoch(model, dl_train_sc4a, criterion, optimizer=optimizer)
    val_loss, val_acc, val_f2, val_spec = run_epoch(model, dl_val_sc4a, criterion, optimizer=None)
    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {tr_loss:.4f} acc {tr_acc:.4f} "
        f"f2 {tr_f2:.4f} spec {tr_spec:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} "
        f"f2 {val_f2:.4f} spec {val_spec:.4f}"
    )

    meets_specificity = val_spec >= MIN_SPECIFICITY

    is_better_unconstrained = (
        val_f2 > best_val_f2_unconstrained + MIN_DELTA
    ) or (
        np.isclose(val_f2, best_val_f2_unconstrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_unconstrained
    )

    if is_better_unconstrained:
        best_val_f2_unconstrained = val_f2
        best_val_loss_unconstrained = val_loss
        best_epoch_unconstrained = epoch
        best_state_unconstrained = copy.deepcopy(model.state_dict())

    is_better_constrained = meets_specificity and (
        val_f2 > best_val_f2_constrained + MIN_DELTA
    ) or (
        meets_specificity
        and np.isclose(val_f2, best_val_f2_constrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_constrained
    )

    if is_better_constrained:
        best_val_f2_constrained = val_f2
        best_val_loss_constrained = val_loss
        best_epoch_constrained = epoch
        best_state_constrained = copy.deepcopy(model.state_dict())

    if best_state_constrained is not None:
        best_state = best_state_constrained
        best_epoch = best_epoch_constrained
        best_val_f2 = best_val_f2_constrained
        best_val_loss = best_val_loss_constrained
        selection_mode = "constrained (specificity met)"
    else:
        best_state = best_state_unconstrained
        best_epoch = best_epoch_unconstrained
        best_val_f2 = best_val_f2_unconstrained
        best_val_loss = best_val_loss_unconstrained

        selection_mode = ("unconstrained (fallback — no epoch met specificity)")


if best_state is not None:
    #model.load_state_dict(best_state)
    torch.save(best_state,BEST_PATH)

    print("\nModel selected via:", selection_mode)
    print(f"Best epoch: {best_epoch} | val_f2: {best_val_f2:.4f} | val_loss: {best_val_loss:.4f}")
else:
    raise RuntimeError("No valid model state was recorded.")

sc4_train_result.append([
    "sc4a-efficientnet-b0-adam",
    "sc4a", "efficientnet-b0", "adam",
    selection_mode,
    best_epoch,
    best_val_f2,
    ])

Optimizer     : Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 0.0001
    maximize: False
    weight_decay: 0.0
)
Loss Function : CrossEntropyLoss()
Loss Weight   : tensor([0.0029, 0.0038], device='cuda:0')
Epoch 01/20 | train loss 0.6398 acc 0.6464 f2 0.6909 spec 0.5833 | val loss 0.7935 acc 0.3158 f2 0.3488 spec 0.2727
Epoch 02/20 | train loss 0.4612 acc 0.9030 f2 0.9049 spec 0.8937 | val loss 0.8687 acc 0.3158 f2 0.2439 spec 0.3636
Epoch 03/20 | train loss 0.2965 acc 0.9622 f2 0.9546 spec 0.9684 | val loss 0.9589 acc 0.4737 f2 0.2632 spec 0.6364
Epoch 04/20 | train loss 0.1461 acc 0.9934 f2 0.9946 spec 0.9914 | val loss 1.1377 acc 0.4737 f2 0.2632 spec 0.6364
Epoch 05/20 | train loss 0.0827 acc 0.9918 f2 0.9939 spec 0.9885 | val loss 1.3286 acc 0.4737 f2 0.1389 spec 0.7273
Epoch 06/20 | train loss 0.051

##Sc.4.a. EfficientNet-B0 with AdamW

###Sc.4.a. EfficientNet-B0 with AdamW Class Weight CE Loss Function

In [30]:
criterion = nn.CrossEntropyLoss(weight=None)

In [31]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
#torch.use_deterministic_algorithms(True)

model = create_model(num_classes=2).to(device)


Device: cuda


In [32]:
# Class Weight
train_counts = train_sc4a_df["label"].value_counts().sort_index().to_dict()
n0 = train_counts.get(0, 1)
n1 = train_counts.get(1, 1)

# Rare class (1) gets HIGH weight, frequent class (0) gets LOW weight
base_class_weights = torch.tensor([1.0 / n0, 1.0 / n1], dtype=torch.float32, device=device)

In [33]:
criterion = nn.CrossEntropyLoss(weight=base_class_weights)

###Sc.4.a. AdamW Optimizer

In [34]:
# Optimizer Parameter Set

LR = 1e-4
BETAS = (0.9, 0.999)
EPS = 1e-8
WEIGHT_DECAY = 1e-2

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    betas=BETAS,
    eps=EPS,
    weight_decay=WEIGHT_DECAY
    )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

###Sc.4.a. EfficientNet-B0 with AdamW Training

In [35]:
# Load Dataset

dl_train_sc4a = make_loader(train_sc4a_df, train_tfms, shuffle=True)
dl_val_sc4a   = make_loader(val_sc4a_df, eval_tfms, shuffle=False)
dl_test_sc4a  = make_loader(test_sc4a_df, eval_tfms, shuffle=False)

print(f"\ntrain={len(train_sc4a_df)} | val={len(val_sc4a_df)} | test={len(test_sc4a_df)}")


train=608 | val=19 | test=19


In [36]:
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc4a_adamw.pth"

In [37]:
# Training Loop

print("Optimizer     :", optimizer)
print("Loss Function :", criterion)
print("Loss Weight   :", criterion.weight)

# ---------------------------------------------------------
# Best-model tracking
# ---------------------------------------------------------

best_val_f2   = -1.0
best_val_loss = float("inf")
best_epoch    = None
best_state    = None

best_val_f2_constrained   = -1.0
best_val_f2_unconstrained = -1.0
best_val_loss_constrained   = float("inf")
best_val_loss_unconstrained = float("inf")
best_state_constrained    = None
best_state_unconstrained  = None
best_epoch_constrained    = None
best_epoch_unconstrained  = None

MIN_SPECIFICITY = 0.30
MIN_DELTA = 1e-4

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_acc, tr_f2, tr_spec = run_epoch(model, dl_train_sc4a, criterion, optimizer=optimizer)
    val_loss, val_acc, val_f2, val_spec = run_epoch(model, dl_val_sc4a, criterion, optimizer=None)
    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {tr_loss:.4f} acc {tr_acc:.4f} "
        f"f2 {tr_f2:.4f} spec {tr_spec:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} "
        f"f2 {val_f2:.4f} spec {val_spec:.4f}"
    )

    meets_specificity = val_spec >= MIN_SPECIFICITY

    is_better_unconstrained = (
        val_f2 > best_val_f2_unconstrained + MIN_DELTA
    ) or (
        np.isclose(val_f2, best_val_f2_unconstrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_unconstrained
    )

    if is_better_unconstrained:
        best_val_f2_unconstrained = val_f2
        best_val_loss_unconstrained = val_loss
        best_epoch_unconstrained = epoch
        best_state_unconstrained = copy.deepcopy(model.state_dict())

    is_better_constrained = meets_specificity and (
        val_f2 > best_val_f2_constrained + MIN_DELTA
    ) or (
        meets_specificity
        and np.isclose(val_f2, best_val_f2_constrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_constrained
    )

    if is_better_constrained:
        best_val_f2_constrained = val_f2
        best_val_loss_constrained = val_loss
        best_epoch_constrained = epoch
        best_state_constrained = copy.deepcopy(model.state_dict())

    if best_state_constrained is not None:
        best_state = best_state_constrained
        best_epoch = best_epoch_constrained
        best_val_f2 = best_val_f2_constrained
        best_val_loss = best_val_loss_constrained
        selection_mode = "constrained (specificity met)"
    else:
        best_state = best_state_unconstrained
        best_epoch = best_epoch_unconstrained
        best_val_f2 = best_val_f2_unconstrained
        best_val_loss = best_val_loss_unconstrained

        selection_mode = ("unconstrained (fallback — no epoch met specificity)")


if best_state is not None:
    #model.load_state_dict(best_state)
    torch.save(best_state,BEST_PATH)

    print("\nModel selected via:", selection_mode)
    print(f"Best epoch: {best_epoch} | val_f2: {best_val_f2:.4f} | val_loss: {best_val_loss:.4f}")
else:
    raise RuntimeError("No valid model state was recorded.")

sc4_train_result.append([
    "sc4a-efficientnet-b0-adamw",
    "sc4a", "efficientnet-b0", "adamw",
    selection_mode,
    best_epoch,
    best_val_f2,
    ])

Optimizer     : AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 0.0001
    maximize: False
    weight_decay: 0.01
)
Loss Function : CrossEntropyLoss()
Loss Weight   : tensor([0.0029, 0.0038], device='cuda:0')
Epoch 01/20 | train loss 0.6398 acc 0.6480 f2 0.6914 spec 0.5862 | val loss 0.7935 acc 0.3158 f2 0.3488 spec 0.2727
Epoch 02/20 | train loss 0.4612 acc 0.9030 f2 0.9049 spec 0.8937 | val loss 0.8687 acc 0.3158 f2 0.2439 spec 0.3636
Epoch 03/20 | train loss 0.2965 acc 0.9622 f2 0.9546 spec 0.9684 | val loss 0.9588 acc 0.4737 f2 0.2632 spec 0.6364
Epoch 04/20 | train loss 0.1461 acc 0.9934 f2 0.9946 spec 0.9914 | val loss 1.1376 acc 0.4737 f2 0.2632 spec 0.6364
Epoch 05/20 | train loss 0.0827 acc 0.9918 f2 0.9939 spec 0.9885 | val loss 1.3284 acc 0.4737 f2 0.1389 spec 0.7273
Epoch 06/20 | train loss 0.05

#Sc.4.b. Model Development

##Sc.4.b. EfficientNet-B0 with SGD

###Sc.4.b. EfficientNet-B0 with SGD Class Weight CE Loss Function

In [38]:
criterion = nn.CrossEntropyLoss(weight=None)

In [39]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
#torch.use_deterministic_algorithms(True)

model = create_model(num_classes=2).to(device)


Device: cuda


In [40]:
# Class Weight
train_counts = train_sc4b_df["label"].value_counts().sort_index().to_dict()
n0 = train_counts.get(0, 1)
n1 = train_counts.get(1, 1)

# Rare class (1) gets HIGH weight, frequent class (0) gets LOW weight
base_class_weights = torch.tensor([1.0 / n0, 1.0 / n1], dtype=torch.float32, device=device)

In [41]:
criterion = nn.CrossEntropyLoss(weight=base_class_weights)

###Sc.4.b. SGD Optimizer

In [42]:
# SGD
LR          = 1e-3
MOMENTUM    = 0.9
WEIGHT_DECAY= 1e-4
NESTEROV    = True

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    nesterov=NESTEROV
    )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

###Sc.4.b. EfficientNet-B0 with SGD Training

In [43]:
# Load Dataset

dl_train_sc4b = make_loader(train_sc4b_df, train_tfms, shuffle=True)
dl_val_sc4b   = make_loader(val_sc4b_df, eval_tfms, shuffle=False)
dl_test_sc4b  = make_loader(test_sc4b_df, eval_tfms, shuffle=False)

print(f"\ntrain={len(train_sc4b_df)} | val={len(val_sc4b_df)} | test={len(test_sc4b_df)}")


train=152 | val=19 | test=19


In [44]:
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc4b_sgd.pth"

In [45]:
# Training Loop

print("Optimizer     :", optimizer)
print("Loss Function :", criterion)
print("Loss Weight   :", criterion.weight)

# ---------------------------------------------------------
# Best-model tracking
# ---------------------------------------------------------

best_val_f2   = -1.0
best_val_loss = float("inf")
best_epoch    = None
best_state    = None

best_val_f2_constrained   = -1.0
best_val_f2_unconstrained = -1.0
best_val_loss_constrained   = float("inf")
best_val_loss_unconstrained = float("inf")
best_state_constrained    = None
best_state_unconstrained  = None
best_epoch_constrained    = None
best_epoch_unconstrained  = None

MIN_SPECIFICITY = 0.30
MIN_DELTA = 1e-4

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_acc, tr_f2, tr_spec = run_epoch(model, dl_train_sc4b, criterion, optimizer=optimizer)
    val_loss, val_acc, val_f2, val_spec = run_epoch(model, dl_val_sc4b, criterion, optimizer=None)
    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {tr_loss:.4f} acc {tr_acc:.4f} "
        f"f2 {tr_f2:.4f} spec {tr_spec:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} "
        f"f2 {val_f2:.4f} spec {val_spec:.4f}"
    )

    meets_specificity = val_spec >= MIN_SPECIFICITY

    is_better_unconstrained = (
        val_f2 > best_val_f2_unconstrained + MIN_DELTA
    ) or (
        np.isclose(val_f2, best_val_f2_unconstrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_unconstrained
    )

    if is_better_unconstrained:
        best_val_f2_unconstrained = val_f2
        best_val_loss_unconstrained = val_loss
        best_epoch_unconstrained = epoch
        best_state_unconstrained = copy.deepcopy(model.state_dict())

    is_better_constrained = meets_specificity and (
        val_f2 > best_val_f2_constrained + MIN_DELTA
    ) or (
        meets_specificity
        and np.isclose(val_f2, best_val_f2_constrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_constrained
    )

    if is_better_constrained:
        best_val_f2_constrained = val_f2
        best_val_loss_constrained = val_loss
        best_epoch_constrained = epoch
        best_state_constrained = copy.deepcopy(model.state_dict())

    if best_state_constrained is not None:
        best_state = best_state_constrained
        best_epoch = best_epoch_constrained
        best_val_f2 = best_val_f2_constrained
        best_val_loss = best_val_loss_constrained
        selection_mode = "constrained (specificity met)"
    else:
        best_state = best_state_unconstrained
        best_epoch = best_epoch_unconstrained
        best_val_f2 = best_val_f2_unconstrained
        best_val_loss = best_val_loss_unconstrained

        selection_mode = ("unconstrained (fallback — no epoch met specificity)")


if best_state is not None:
    #model.load_state_dict(best_state)
    torch.save(best_state,BEST_PATH)

    print("\nModel selected via:", selection_mode)
    print(f"Best epoch: {best_epoch} | val_f2: {best_val_f2:.4f} | val_loss: {best_val_loss:.4f}")
else:
    raise RuntimeError("No valid model state was recorded.")

sc4_train_result.append([
    "sc4b-efficientnet-b0-sgd",
    "sc4b", "efficientnet-b0", "sgd",
    selection_mode,
    best_epoch,
    best_val_f2,
    ])

Optimizer     : SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.001
    maximize: False
    momentum: 0.9
    nesterov: True
    weight_decay: 0.0001
)
Loss Function : CrossEntropyLoss()
Loss Weight   : tensor([0.0115, 0.0154], device='cuda:0')
Epoch 01/20 | train loss 0.6858 acc 0.5263 f2 0.6510 spec 0.3793 | val loss 0.7019 acc 0.5263 f2 0.5814 spec 0.4545
Epoch 02/20 | train loss 0.6807 acc 0.5395 f2 0.5908 spec 0.4713 | val loss 0.7428 acc 0.4737 f2 0.2632 spec 0.6364
Epoch 03/20 | train loss 0.6710 acc 0.5724 f2 0.5769 spec 0.5517 | val loss 0.7571 acc 0.3684 f2 0.1316 spec 0.5455
Epoch 04/20 | train loss 0.6643 acc 0.5855 f2 0.5689 spec 0.5862 | val loss 0.7632 acc 0.3158 f2 0.1282 spec 0.4545
Epoch 05/20 | train loss 0.6537 acc 0.6316 f2 0.6490 spec 0.5977 | val loss 0.7607 acc 0.2632 f2 0.0000 spec 0.4545
Epoch 06/20 | train loss 0.6470 acc 0.6053 f2 0.5140 spec 0.6782 | val loss 0.7581 acc 0.2

##Sc.4.b. EfficientNet-B0 with Adam

###Sc.4.b. EfficientNet-B0 with Adam Class Weight CE Loss Function

In [46]:
criterion = nn.CrossEntropyLoss(weight=None)

In [47]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
#torch.use_deterministic_algorithms(True)

model = create_model(num_classes=2).to(device)


Device: cuda


In [48]:
# Class Weight
train_counts = train_sc4b_df["label"].value_counts().sort_index().to_dict()
n0 = train_counts.get(0, 1)
n1 = train_counts.get(1, 1)

# Rare class (1) gets HIGH weight, frequent class (0) gets LOW weight
base_class_weights = torch.tensor([1.0 / n0, 1.0 / n1], dtype=torch.float32, device=device)

In [49]:
criterion = nn.CrossEntropyLoss(weight=base_class_weights)

###Sc.4.b. Adam Optimizer

In [50]:
# Adam
LR          = 1e-4
BETAS       = (0.9, 0.999)
EPS         = 1e-8
WEIGHT_DECAY= 0.0

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    betas=BETAS,
    eps=EPS,
    weight_decay=WEIGHT_DECAY
    )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

###Sc.4.b. EfficientNet-B0 with Adam Training

In [51]:
# Load Dataset

dl_train_sc4b = make_loader(train_sc4b_df, train_tfms, shuffle=True)
dl_val_sc4b   = make_loader(val_sc4b_df, eval_tfms, shuffle=False)
dl_test_sc4b  = make_loader(test_sc4b_df, eval_tfms, shuffle=False)

print(f"\ntrain={len(train_sc4b_df)} | val={len(val_sc4b_df)} | test={len(test_sc4b_df)}")


train=152 | val=19 | test=19


In [52]:
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc4b_adam.pth"

In [53]:
# Training Loop

print("Optimizer     :", optimizer)
print("Loss Function :", criterion)
print("Loss Weight   :", criterion.weight)

# ---------------------------------------------------------
# Best-model tracking
# ---------------------------------------------------------

best_val_f2   = -1.0
best_val_loss = float("inf")
best_epoch    = None
best_state    = None

best_val_f2_constrained   = -1.0
best_val_f2_unconstrained = -1.0
best_val_loss_constrained   = float("inf")
best_val_loss_unconstrained = float("inf")
best_state_constrained    = None
best_state_unconstrained  = None
best_epoch_constrained    = None
best_epoch_unconstrained  = None

MIN_SPECIFICITY = 0.30
MIN_DELTA = 1e-4

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_acc, tr_f2, tr_spec = run_epoch(model, dl_train_sc4b, criterion, optimizer=optimizer)
    val_loss, val_acc, val_f2, val_spec = run_epoch(model, dl_val_sc4b, criterion, optimizer=None)
    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {tr_loss:.4f} acc {tr_acc:.4f} "
        f"f2 {tr_f2:.4f} spec {tr_spec:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} "
        f"f2 {val_f2:.4f} spec {val_spec:.4f}"
    )

    meets_specificity = val_spec >= MIN_SPECIFICITY

    is_better_unconstrained = (
        val_f2 > best_val_f2_unconstrained + MIN_DELTA
    ) or (
        np.isclose(val_f2, best_val_f2_unconstrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_unconstrained
    )

    if is_better_unconstrained:
        best_val_f2_unconstrained = val_f2
        best_val_loss_unconstrained = val_loss
        best_epoch_unconstrained = epoch
        best_state_unconstrained = copy.deepcopy(model.state_dict())

    is_better_constrained = meets_specificity and (
        val_f2 > best_val_f2_constrained + MIN_DELTA
    ) or (
        meets_specificity
        and np.isclose(val_f2, best_val_f2_constrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_constrained
    )

    if is_better_constrained:
        best_val_f2_constrained = val_f2
        best_val_loss_constrained = val_loss
        best_epoch_constrained = epoch
        best_state_constrained = copy.deepcopy(model.state_dict())

    if best_state_constrained is not None:
        best_state = best_state_constrained
        best_epoch = best_epoch_constrained
        best_val_f2 = best_val_f2_constrained
        best_val_loss = best_val_loss_constrained
        selection_mode = "constrained (specificity met)"
    else:
        best_state = best_state_unconstrained
        best_epoch = best_epoch_unconstrained
        best_val_f2 = best_val_f2_unconstrained
        best_val_loss = best_val_loss_unconstrained

        selection_mode = ("unconstrained (fallback — no epoch met specificity)")


if best_state is not None:
    #model.load_state_dict(best_state)
    torch.save(best_state,BEST_PATH)

    print("\nModel selected via:", selection_mode)
    print(f"Best epoch: {best_epoch} | val_f2: {best_val_f2:.4f} | val_loss: {best_val_loss:.4f}")
else:
    raise RuntimeError("No valid model state was recorded.")

sc4_train_result.append([
    "sc4b-efficientnet-b0-adam",
    "sc4b", "efficientnet-b0", "adam",
    selection_mode,
    best_epoch,
    best_val_f2,
    ])

Optimizer     : Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 0.0001
    maximize: False
    weight_decay: 0.0
)
Loss Function : CrossEntropyLoss()
Loss Weight   : tensor([0.0115, 0.0154], device='cuda:0')
Epoch 01/20 | train loss 0.6866 acc 0.4934 f2 0.6111 spec 0.3563 | val loss 0.6998 acc 0.4211 f2 0.4651 spec 0.3636
Epoch 02/20 | train loss 0.5949 acc 0.7368 f2 0.8166 spec 0.6322 | val loss 0.7461 acc 0.5263 f2 0.4878 spec 0.5455
Epoch 03/20 | train loss 0.5444 acc 0.8421 f2 0.9038 spec 0.7586 | val loss 0.7773 acc 0.4737 f2 0.5682 spec 0.3636
Epoch 04/20 | train loss 0.4775 acc 0.9342 f2 0.9517 spec 0.9080 | val loss 0.7952 acc 0.4211 f2 0.4651 spec 0.3636
Epoch 05/20 | train loss 0.4175 acc 0.9474 f2 0.9760 spec 0.9080 | val loss 0.8033 acc 0.3684 f2 0.3571 spec 0.3636
Epoch 06/20 | train loss 0.366

##Sc.4.b. EfficientNet-B0 with AdamW

###Sc.4.b. EfficientNet-B0 with AdamW Class Weight CE Loss Function

In [54]:
criterion = nn.CrossEntropyLoss(weight=None)

In [55]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
#torch.use_deterministic_algorithms(True)

model = create_model(num_classes=2).to(device)


Device: cuda


In [56]:
# Class Weight
train_counts = train_sc4b_df["label"].value_counts().sort_index().to_dict()
n0 = train_counts.get(0, 1)
n1 = train_counts.get(1, 1)

# Rare class (1) gets HIGH weight, frequent class (0) gets LOW weight
base_class_weights = torch.tensor([1.0 / n0, 1.0 / n1], dtype=torch.float32, device=device)

In [57]:
criterion = nn.CrossEntropyLoss(weight=base_class_weights)

###Sc.4.b. AdamW Optimizer

In [58]:
# Optimizer Parameter Set

LR = 1e-4
BETAS = (0.9, 0.999)
EPS = 1e-8
WEIGHT_DECAY = 1e-2

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    betas=BETAS,
    eps=EPS,
    weight_decay=WEIGHT_DECAY
    )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

###Sc.4.b. EfficientNet-B0 with AdamW Training

In [59]:
# Load Dataset

dl_train_sc4b = make_loader(train_sc4b_df, train_tfms, shuffle=True)
dl_val_sc4b   = make_loader(val_sc4b_df, eval_tfms, shuffle=False)
dl_test_sc4b  = make_loader(test_sc4b_df, eval_tfms, shuffle=False)

print(f"\ntrain={len(train_sc4b_df)} | val={len(val_sc4b_df)} | test={len(test_sc4b_df)}")


train=152 | val=19 | test=19


In [60]:
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc4b_adamw.pth"

In [61]:
# Training Loop

print("Optimizer     :", optimizer)
print("Loss Function :", criterion)
print("Loss Weight   :", criterion.weight)

# ---------------------------------------------------------
# Best-model tracking
# ---------------------------------------------------------

best_val_f2   = -1.0
best_val_loss = float("inf")
best_epoch    = None
best_state    = None

best_val_f2_constrained   = -1.0
best_val_f2_unconstrained = -1.0
best_val_loss_constrained   = float("inf")
best_val_loss_unconstrained = float("inf")
best_state_constrained    = None
best_state_unconstrained  = None
best_epoch_constrained    = None
best_epoch_unconstrained  = None

MIN_SPECIFICITY = 0.30
MIN_DELTA = 1e-4

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_acc, tr_f2, tr_spec = run_epoch(model, dl_train_sc4b, criterion, optimizer=optimizer)
    val_loss, val_acc, val_f2, val_spec = run_epoch(model, dl_val_sc4b, criterion, optimizer=None)
    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {tr_loss:.4f} acc {tr_acc:.4f} "
        f"f2 {tr_f2:.4f} spec {tr_spec:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} "
        f"f2 {val_f2:.4f} spec {val_spec:.4f}"
    )

    meets_specificity = val_spec >= MIN_SPECIFICITY

    is_better_unconstrained = (
        val_f2 > best_val_f2_unconstrained + MIN_DELTA
    ) or (
        np.isclose(val_f2, best_val_f2_unconstrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_unconstrained
    )

    if is_better_unconstrained:
        best_val_f2_unconstrained = val_f2
        best_val_loss_unconstrained = val_loss
        best_epoch_unconstrained = epoch
        best_state_unconstrained = copy.deepcopy(model.state_dict())

    is_better_constrained = meets_specificity and (
        val_f2 > best_val_f2_constrained + MIN_DELTA
    ) or (
        meets_specificity
        and np.isclose(val_f2, best_val_f2_constrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_constrained
    )

    if is_better_constrained:
        best_val_f2_constrained = val_f2
        best_val_loss_constrained = val_loss
        best_epoch_constrained = epoch
        best_state_constrained = copy.deepcopy(model.state_dict())

    if best_state_constrained is not None:
        best_state = best_state_constrained
        best_epoch = best_epoch_constrained
        best_val_f2 = best_val_f2_constrained
        best_val_loss = best_val_loss_constrained
        selection_mode = "constrained (specificity met)"
    else:
        best_state = best_state_unconstrained
        best_epoch = best_epoch_unconstrained
        best_val_f2 = best_val_f2_unconstrained
        best_val_loss = best_val_loss_unconstrained

        selection_mode = ("unconstrained (fallback — no epoch met specificity)")


if best_state is not None:
    #model.load_state_dict(best_state)
    torch.save(best_state,BEST_PATH)

    print("\nModel selected via:", selection_mode)
    print(f"Best epoch: {best_epoch} | val_f2: {best_val_f2:.4f} | val_loss: {best_val_loss:.4f}")
else:
    raise RuntimeError("No valid model state was recorded.")

sc4_train_result.append([
    "sc4b-efficientnet-b0-adamw",
    "sc4b", "efficientnet-b0", "adamw",
    selection_mode,
    best_epoch,
    best_val_f2,
    ])

Optimizer     : AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 0.0001
    maximize: False
    weight_decay: 0.01
)
Loss Function : CrossEntropyLoss()
Loss Weight   : tensor([0.0115, 0.0154], device='cuda:0')
Epoch 01/20 | train loss 0.6866 acc 0.4934 f2 0.6111 spec 0.3563 | val loss 0.6998 acc 0.4211 f2 0.4651 spec 0.3636
Epoch 02/20 | train loss 0.5949 acc 0.7368 f2 0.8166 spec 0.6322 | val loss 0.7461 acc 0.5263 f2 0.4878 spec 0.5455
Epoch 03/20 | train loss 0.5444 acc 0.8421 f2 0.9038 spec 0.7586 | val loss 0.7773 acc 0.4737 f2 0.5682 spec 0.3636
Epoch 04/20 | train loss 0.4775 acc 0.9342 f2 0.9517 spec 0.9080 | val loss 0.7952 acc 0.3684 f2 0.4545 spec 0.2727
Epoch 05/20 | train loss 0.4175 acc 0.9474 f2 0.9760 spec 0.9080 | val loss 0.8033 acc 0.3684 f2 0.3571 spec 0.3636
Epoch 06/20 | train loss 0.36

#Result

In [62]:
sc4_train_result_df = pd.DataFrame(sc4_train_result,
                                   columns=['model_name',
                                            'scheme', 'model', 'optimizer',
                                            'selection_mode',
                                            'best_epoch',
                                            'best_val_f2',
                                            ])
display(sc4_train_result_df)

base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/result/"
sc4_train_result_df.to_excel(base + 'sc4_train_result.xlsx', index=False)

,model_name,scheme,model,optimizer,selection_mode,best_epoch,best_val_f2
0,sc4a-efficientnet-b0-sgd,sc4a,efficientnet-b0,sgd,constrained (specificity met),12,0.263158
1,sc4a-efficientnet-b0-adam,sc4a,efficientnet-b0,adam,constrained (specificity met),3,0.263158
2,sc4a-efficientnet-b0-adamw,sc4a,efficientnet-b0,adamw,constrained (specificity met),3,0.263158
3,sc4b-efficientnet-b0-sgd,sc4b,efficientnet-b0,sgd,constrained (specificity met),1,0.581395
4,sc4b-efficientnet-b0-adam,sc4b,efficientnet-b0,adam,constrained (specificity met),3,0.568182
5,sc4b-efficientnet-b0-adamw,sc4b,efficientnet-b0,adamw,constrained (specificity met),3,0.568182
